# LoRA Fine-Tuning with Unsloth

Fine-tune a small LLM using LoRA for efficient training.

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

## 1. Load Model with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print(f'Model loaded: {model.config.model_type}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 2. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.2f}%)')

## 3. Prepare Dataset

In [ ]:
dataset = load_dataset('yahma/alpaca-cleaned', split='train[:1000]')
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}

### Input:
{}"""

def formatting_prompts_func(examples):
    instructions = examples['instruction']
    inputs = examples['input']
    outputs = examples['output']
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, output, input)
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'Dataset size: {len(dataset)}')

## 4. Training

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs/lora-finetune',
    ),
)
# trainer.train()  # Uncomment to run training

## 5. Save & Inference

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{'role': 'user', 'content': 'What is machine learning?'}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True)
response = tokenizer.batch_decode(outputs)
print(response[0])